# Thinkin' bout those automata

I really want to create agents that engage in musical decisionmaking on their own. Of course, they would be trained. Ultimately, I, the human, have to make my subjective human judgments about things to teach the computer how to model my judgments.

## Brainstorming a way of doing this

This is a tough nut to crack, because there's so much complexity involved in the unconscious processes of me playing and writing music.

Certain features have to be accounted for in order to make the kinds of musical decisions I make. 

For one, I write with respect to sections, and harmonic rhythms, and chord progressions, and the way the notes groove, etc. 

## Groove

A groove is interesting. There's the meter, and everything becomes a fragmented expression of the meter. What classifies as "up beats" and "down beats" is pretty subjective. It's one of those things like dissonance. But you can anything subjective like that can be modeled - you just have to find your parameters and tune them so the model fits the perspective. I have long thought of hypothetical models that store a dictionary containing all the choices you can make, and mapping them to different quality judgments. So for example, certain chords in the dictionary are registered as dissonant, some as consonant, etc, and the entire "association table" constitutes the subjective model. It would be cool if this model were trained on input from the user, asking the user to score different chords and such.

## Anyways, chords

There has to be some process that chord progressions are generated. 

It seems to me the way to do it is to start with a section. You can...

- Add another section
- Extend/multiply the section
- Contract/divide the section

So, you start with the A section. It's 1 bar long to start. 
You can multiply it by 8, now it's 8 bars. 
Then you can split it into two sections, 4 bars each. Now you have an A and B section.

I mean, you can basically just use Clips for this, right? 

So, what do you do inside of the section?
Another feature is harmonic rhythm: how often the chord/scale changes.
What you can do is just treat it as a space to split up and be looped. Kind of like tidalcycles.

To start, imagine you just have a looping chord progression: 1 bar, C major triad. Wow, it sounds incredible!
But imagine you want to add another chord. You could create a 1 bar clip with a G major triad in it, and concatenate it to the end of the section.

Now you have a 2 bar long section that goes C maj for 1 bar then G maj for 1 bar.

You could take a harmonic rhythm that goes (in bars) 1, 1, 1, 1. You can then attach a chord sequence to it. 

There are _so_ many ways to fit a chord sequence to this looping rhythm. 

When building a chord progression, you could start by building a bass line. For example, C G Ab Bb. 

It's good to generate this stuff using interval class cycles. And if it's for a looping part, then it makes sense to use a cycle with a net transposition of 0. 
This cycle is 7 1 2 2. Because the sum of the interval classes (mod 12) is 0, we restart on the same pitch class every time.

From there, we decide what chords to build on these bass notes. 

We could just decide on a chord type for each root. Very non-functional harmonic approach. So, major for the first two, and minor for the second two.
This would give us C, G, Abm, Bbm. Weird. 

Another process I also use is assigning a flavor (min, maj, etc) to one, and then imposing a matching scale, finding chords in the same scale as the first chord which contain the next bass note, etc. 

It's often not the case that every chord fits into a single pitch class collection. 
Like, consider if they were all major chords: C, G, Ab, Bb. That collection is C D Eb E F G Ab Bb B. Now that's a funky 9-note scale!

But indeed: there can be a process where you iterate over the bass notes and build chords off them. 
But...

- How long do you iterate?
- How do you decide on a scale to take a chord from?

Wrt that last question: you have to get a list of scales that contain the previous chord AND the next bass note. But there will likely be multiple.

So the question remains: how do you choose?

I guess it all comes back to weighted random choice. 

## The task at hand

Create an algorithm that takes a bass line in the form of a pitch class sequence, and builds up a chord progression in the form of a pitch class set sequence.
No harmonic rhythm or anything like that: we're just building up a pitch class set sequence from a pitch class sequence. 

This is a non-deterministic function.

In [ ]:
import random
from typing import Iterator

from harmonica.find._find_pcset import (
    find_pcset_supersets,
    find_pcsets_containing_pclass,
)
from harmonica.pitch._changes import PCSetSeq
from harmonica.pitch._melody import PCSequence
from harmonica.pitch._scales import PitchClassSet

# Our bass line
bassline = PCSequence([0, 7, 8, 10], 12)


def interesting_pcsets(subset: PitchClassSet) -> Iterator[PitchClassSet]:
    # Returns an iterator of interesting pitch class sets
    pcsets = find_pcset_supersets(subset)
    pcsets = filter(lambda pcset: pcset.cardinality <= 4, pcsets)
    pcsets = filter(lambda pcset: pcset.cardinality >= subset.cardinality, pcsets)
    return pcsets


def build_progression_from_bassline(bass_line: PCSequence) -> PCSetSeq:
    # Initialize pcset sequence
    progression: list[PitchClassSet] = []

    # Bind list of all pitch class sets (mod 12) to variable
    all_pcsets = list(interesting_pcsets(PitchClassSet([bass_line[0]], 12)))

    # Choose pitch class set that contains first pitch class in bass_line
    first_pcset: PitchClassSet = random.choice(all_pcsets)
    progression.append(first_pcset)

    # Loop through the rest of the bass notes
    for bass_note in bass_line[1:]:
        # Get a list of all pcsets that contain the previous chord and the next bass note
        pclasses = progression[-1].pitch_classes
        if bass_note not in pclasses:
            pclasses.append(bass_note)
        pclasses.sort()

        all_pcsets = list(interesting_pcsets(PitchClassSet(pclasses, 12)))

        next_pcset: PitchClassSet = random.choice(all_pcsets)
        print(next_pcset)
        progression.append(next_pcset)

    return PCSetSeq(progression)


progression = build_progression_from_bassline(bassline)

print()
print("SUGHSJGHG")
for pcset in progression:
    print(pcset)

This approach is a bit of a disaster. It keeps filtering pcsets through predicates that nothing can satisfy, yielding empty iterators. So I have to figure out how to handle those situations.

What specifically was failing here?

First, it passes the pitch class set {0} mod 12 to get the first pcset. It finds all the supersets of the set, aka all the pcsets that contain 0. Then it keeps the ones with between 1 and 8 pitch classes. It then selects a random one and adds it to the progression.

That step always works fine. It's the loop that comes next that fucks things up.

It grabs the next bass note, and combines it with the pitch classes of the previous chord in the progression. The problem is this: if the last chord contained the max number of pitches, then there's a chance the combined pcset will contain one more than the max that's generated. 

So, how do we fix this? Well, how do I normally deal with this? 

I typically don't make chords with more than 4 or 5 notes. So... idk.
When I generate a chord with 4 notes, and I add the bass note to get a 5-note chord, what do I do? 

In the routine that generates and filters the pcset iterator, I could add logic that ensures that notes don't go over the max. 

...Also, am I actually implementing my original idea correctly? 

## WTF was the original idea?

The original idea was that, whenever you figure out what chord to place next, you take the previous chord in the progression and the next bass note, and you derive a scale that the notes fit into. Here's the machinery I skipped: I have to derive the uhh... what's the name I gave it? IDK you take the indices of the previous chord's pitch classes in the pitch class set you fit to. 


OKAY! Here's an example. 

The previous chord is [0,4,7,11], and the next bass note is 2. So you find all scales that contain [0,2,4,7,11]. You pick one: {0,2,4,6,7,9,11}. 

Now you find the indices of [0,4,7,11] in {0,2,4,6,7,9,11}. They are [0,2,4,6]. 
The index of the next bass note, 2, is 1. 

Okay, now here's what you do: you transpose the indices. This is how you move a chord shape around _within_ a scale, walking through the chords in a scale. 

You transpose the indices until one of them is equal to the index of the next bass note, 1.

Thus, the indices you end up with will be [0,1,3,5], [1,3,5,6], [1,3,4,6], or [1,2,4,6].

Now you just pass the indices, such as [1,3,4,6], through the scale {0,2,4,6,7,9,11}.
This yields {2,6,7,11}.

If this were the progression building routine, then you would have the same size chord every time.

How about we try this again?

In [ ]:
import random
from typing import Iterator

from harmonica.find._find_pcset import (
    find_pcset_supersets,
    find_pcsets_containing_pclass,
)
from harmonica.pitch._changes import PCSetSeq
from harmonica.pitch._melody import PCSequence
from harmonica.pitch._scales import PitchClassSet

# Our bass line
bassline = PCSequence([0, 7, 8, 10], 12)


def build_progression_from_bassline_v2(bass_line: PCSequence) -> PCSetSeq:
    progression: list[PitchClassSet] = []

    ### Place first chord ###
    # Picks random chord containing the first bass note that is between 3 and 5 notes
    first_bass_note = bass_line[0]
    pcsets_to_choose = find_pcsets_containing_pclass(first_bass_note, 12)
    pcsets_to_choose = filter(
        lambda pcset: pcset.cardinality >= 3 and pcset.cardinality <= 4,
        pcsets_to_choose,
    )
    first_pcset = random.choice(list(pcsets_to_choose))
    first_pcset.root = first_bass_note
    progression.append(first_pcset)

    ### Place next chords ###
    # Each chord is based on the last
    for next_bass_note in bass_line[1:]:
        prev_pcset = progression[-1]

        # Start by combining pitch classes of bass_note and prev_pcset
        pclasses = list(sorted(
            set(prev_pcset.pitch_classes).union(set([next_bass_note]))
        ))

        # Choose scale with 7-9 notes that fits these pitch classes
        pcsets_to_choose = find_pcset_supersets(PitchClassSet(pclasses, 12))
        pcsets_to_choose = filter(
            lambda pcset: pcset.cardinality >= 7 and pcset.cardinality <= 9,
            pcsets_to_choose,
        )
        scale_of_next_chord: PitchClassSet = random.choice(list(pcsets_to_choose))

        # Get indices of previous chord in scale and bass note
        chord_indices = [scale_of_next_chord.index(pclass) for pclass in prev_pcset]
        bass_index = scale_of_next_chord.index(next_bass_note)

        # Get list of all transpositions of indices that contain the bass index
        indices_choices = []
        scale_card = scale_of_next_chord.cardinality
        for i in range(scale_card):
            transposed_indices = [(index + i) % scale_card for index in chord_indices]
            if bass_index in transposed_indices:
                indices_choices.append(transposed_indices)

        # Randomly choose a set of indices
        chosen_indices = random.choice(indices_choices)

        next_pcset = PitchClassSet(
            pitch_classes=sorted([scale_of_next_chord[i] for i in chosen_indices]),
            modulus=12,
            root=next_bass_note,
        )

        progression.append(next_pcset)

    return PCSetSeq(progression)


bassline = PCSequence([0, 7, 8, 10], 12)
progression = build_progression_from_bassline_v2(bassline)

for chord in progression:
    print(chord)